<a href="https://colab.research.google.com/github/maick-code/AIMS-Capstone/blob/arena%2F01a03c0c-aims-capstone/VaxiMere_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# VaxiMère-QA-CG — Pipeline + entraînement (Google Colab)

Dataset d'intentions multilingue (français `fra`, lingala `lin`, kituba `mkw`) sur
la **vaccination pédiatrique au Congo-Brazzaville** : génération, puis fine-tuning
d'un classifieur d'intentions (cible finale : Gemma 3 + LoRA).

**Étapes** :
1. Installation des dépendances
2. Vérification du GPU (T4)
3. Récupération du code (clone automatique)
4. Validation hors-ligne (sans modèle, rapide)
5. Génération du dataset v2 (~2 400 exemples)
6. **Phase 1** — classifieur encodeur `cis-lmu/glot500-base`
7. **Phase 2** — petit LLM + LoRA `Qwen/Qwen2.5-0.5B-Instruct`
8. Publications Hugging Face (dataset + modèle) — optionnel
9. Inférence rapide

> Exécutez les cellules **dans l'ordre**. La cellule 5 (génération du dataset)
> requiert le GPU et l'accès à Hugging Face (télécharge mDeBERTa + NLLB-600M).


In [ ]:
# 1) Installation des dépendances (Colab fournit déjà torch/transformers)
!pip install -q datasets transformers pandas accelerate sentencepiece huggingface_hub \
  peft trl bitsandbytes scikit-learn


In [ ]:
# 2) Vérification du GPU (un T4 suffit largement)
!nvidia-smi -L
import torch
print("CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# 3) Récupération du code — clone AUTOMATIQUE (avec fallback)
import os, subprocess
from pathlib import Path
from IPython import get_ipython

REPO_URL = "https://github.com/maick-code/AIMS-Capstone.git"
BRANCH   = "arena/01a03c0c-aims-capstone"
REPO_DIR = Path("/content/AIMS-Capstone")

def find_repo():
    for c in (REPO_DIR, Path("/content"), Path.cwd()):
        if (c / "run_pipeline.py").exists() and (c / "vaximere").exists():
            return c
    return None

repo = find_repo()
if repo is None:
    for args in (["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
                 ["git", "clone", REPO_URL, str(REPO_DIR)]):
        r = subprocess.run(args, capture_output=True, text=True)
        if r.returncode == 0:
            break
    repo = find_repo()

if repo is None:
    print("⚠️ Échec du clonage automatique. Uploadez manuellement le dépôt, puis relancez.")
    repo = Path.cwd()
else:
    get_ipython().run_line_magic("cd", str(repo))
    print("✅ Dépôt prêt :", repo)

print("Répertoire de travail :", os.getcwd())


In [ ]:
# 4) Validation hors-ligne (sans modèle ni réseau, rapide)
import os
if os.path.exists("/content/AIMS-Capstone/run_pipeline.py"):
    os.chdir("/content/AIMS-Capstone")

!python selftest.py
!python selftest_split.py


In [ ]:
# 5) Test du câblage du pipeline (dryrun : aucun modèle téléchargé)
#    Sorties : data/dryrun/
import os
if os.path.exists("/content/AIMS-Capstone/run_pipeline.py"):
    os.chdir("/content/AIMS-Capstone")

!python run_pipeline.py --mode dryrun


In [ ]:
# 6) Génération du dataset v2 (~2 400 exemples) — COMPLET
#    Télécharge mDeBERTa-v3 + NLLB-600M ; ~10-20 min sur T4.
#    Sorties : data/final/vaximere_qa_cg_train.{jsonl,csv}, faq_validee.json, stats_report.json
import os
if os.path.exists("/content/AIMS-Capstone/run_pipeline.py"):
    os.chdir("/content/AIMS-Capstone")

!python run_pipeline.py --mode full


In [ ]:
# 7) Inspection des sorties (data/final/)
import json
from pathlib import Path

d = Path("data/final")
if not d.exists():
    print("⚠️ data/final/ absent : lancez d'abord la cellule 6 (--mode full).")
else:
    for f in sorted(d.glob("*.jsonl")):
        n = sum(1 for _ in f.open(encoding="utf-8"))
        first = next(f.open(encoding="utf-8"), "").strip()
        print(f"{f.name}: {n} lignes | aperçu -> {first[:160]}")
    stats = d / "stats_report.json"
    if stats.exists():
        s = json.load(stats.open(encoding="utf-8"))
        print(json.dumps(s, ensure_ascii=False, indent=2)[:1200])


In [ ]:
# 8) PHASE 1 — Classifieur encodeur (rapide : fp16 + sauvegarde finale uniquement)
#     Backbone : glot500-base (couvre fra + lingala + kikongo/kituba).
!python vaximere/training/train_encoder.py \
    --jsonl data/final/vaximere_qa_cg_train.jsonl \
    --model-name cis-lmu/glot500-base \
    --epochs 8 --batch-size 32 --lr 5e-5 \
    --out outputs/encoder


In [ ]:
# 9) PHASE 2 — Petit LLM + LoRA (QLoRA 4-bit, ~10-20 min sur T4)
#     Modèle par défaut : Qwen/Qwen2.5-0.5B-Instruct (NON gated, Apache-2.0).
#     ⚠️ Gemma/Llama sont « gated » : accepter leur licence sur huggingface.co d'abord.
!python vaximere/training/train_decoder.py \
    --jsonl data/final/vaximere_qa_cg_train.jsonl \
    --model-name Qwen/Qwen2.5-0.5B-Instruct \
    --epochs 3 --batch-size 2 \
    --out outputs/decoder_lora


In [ ]:
# 10) Pousser le modèle encodeur sur le Hub (token masqué) — optionnel
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Token HF (write) : ")

!python vaximere/training/train_encoder.py \
    --jsonl data/final/vaximere_qa_cg_train.jsonl \
    --model-name cis-lmu/glot500-base \
    --out outputs/encoder \
    --push-only --hub-model-id Semence/vaximere-intent-glot500


In [ ]:
# 11) Pousser le dataset vers le Hugging Face Hub — optionnel
#     Token "write" : https://huggingface.co/settings/tokens
import os, subprocess, sys
from getpass import getpass

os.chdir("/content/AIMS-Capstone")
REPO_ID = "Semence/vaximere-qa-cg"   # <-- adaptez : votre-org/nom-du-dataset
os.environ["HF_TOKEN"] = getpass("Collez votre token Hugging Face (write) : ")

r = subprocess.run([sys.executable, "push_to_hub.py",
                    "--repo-id", REPO_ID,
                    "--jsonl", "data/final/vaximere_qa_cg_train.jsonl",
                    "--faq", "data/final/faq_validee.json",
                    "--stats", "data/final/stats_report.json"])
print("✅ Dataset publié : https://huggingface.co/datasets/" + REPO_ID if r.returncode == 0
      else f"❌ Échec du push (code {r.returncode}). Voir l'erreur ci-dessus.")


In [ ]:
# 12) Inférence rapide avec le modèle encodeur entraîné
from pathlib import Path

model_path = Path("outputs/encoder/model")
if not model_path.exists():
    print("❌ Le modèle n'existe pas : lancez la cellule 8 (Phase 1) et vérifiez "
          "qu'elle se termine SANS erreur (regardez les logs au-dessus).")
else:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    import torch

    tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    model = AutoModelForSequenceClassification.from_pretrained(str(model_path))
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    exemples = [
        "Mon bébé a de la fièvre après le vaccin, est-ce normal ?",
        "On dit que le vaccin rend les enfants stériles, est-ce vrai ?",
        "Mwana na ngai azali na fièvre nsima ya vaccin, ezali malamu ?",
        "Sambu na nki kupesa mwana na mono vaccine ?",
    ]
    for t in exemples:
        enc = tokenizer(t, return_tensors="pt", truncation=True, max_length=128).to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits
        pred = model.config.id2label[int(logits.argmax(-1))]
        print(f"{t[:60]:<62} -> {pred}")


In [ ]:
# 13) Téléchargement des livrables (optionnel)
from google.colab import files
from pathlib import Path

for p in [Path("data/final/vaximere_qa_cg_train.jsonl"),
          Path("data/final/vaximere_qa_cg_train.csv"),
          Path("data/final/faq_validee.json"),
          Path("data/final/stats_report.json")]:
    if p.exists():
        files.download(str(p))
    else:
        print("(absent)", p)
